# Training

## Imports

In [87]:
import torch
import pandas as pd
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import logging
from pathlib import Path

import utils.data.dataset as dataset
import models.sequence_classifier as sequence_classifier
import config.experiments as exp_config
import utils.experiments.loss_factory as loss_factory
import utils.experiments.optimizer_factory as optimizer_factory
import utils.experiments.sanity_checks as sanity_checks
import utils.experiments.training as training

import importlib

importlib.reload(dataset)
importlib.reload(sequence_classifier)
importlib.reload(exp_config)
importlib.reload(loss_factory)
importlib.reload(optimizer_factory)
importlib.reload(sanity_checks)
importlib.reload(training)

<module 'utils.experiments.training' from 'F:\\Software\\Dataspell\\AdvertisementGroupClassification\\utils\\experiments\\training.py'>

## Random Seed

In [88]:
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)

##  Load Data

In [89]:
DATA_FOLDER_PATH = "../data/processed/"

TRAIN_PATH = DATA_FOLDER_PATH + "train.csv"
VAL_PATH = DATA_FOLDER_PATH + "val.csv"
TEST_PATH = DATA_FOLDER_PATH + "test.csv"

TARGET_COL = "group_id_encoded"
SEQ_COL = "name_processed"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

In [90]:
X_train = train_df[SEQ_COL].values
y_train = train_df[TARGET_COL].values

X_val = val_df[SEQ_COL].values
y_val = val_df[TARGET_COL].values

X_test = test_df[SEQ_COL].values
y_test = test_df[TARGET_COL].values

NUM_CLASSES = len(set(y_train))

## Config

In [91]:
experiment_name = "baseline"

In [92]:
cfg = exp_config.EXPERIMENTS[experiment_name]

model_cfg = cfg["model"]
train_cfg = cfg["training"]
optimizer_cfg = cfg["optimizer"]
loss_cfg = cfg["loss"]

## Tokenizer

In [93]:
MODEL_TOKENIZER_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_TOKENIZER_NAME)

In [94]:
all_texts = list(X_train) + list(X_val) + list(X_test)
max_tokens = max(
    len(tokenizer.encode(text, add_special_tokens=True))
    for text in all_texts
)

max_length = 1 << (max_tokens - 1).bit_length() # next power of 2
print(f"Longest sequence: {max_tokens} tokens → using max_length={max_length}")

Longest sequence: 39 tokens → using max_length=64


## Dataset and DataLoader

In [95]:
BATCH_SIZE = train_cfg["batch_size"]

In [96]:
train_dataset = dataset.SequenceClassificationDataset(X_train, y_train, tokenizer, max_length=max_length)
val_dataset   = dataset.SequenceClassificationDataset(X_val,   y_val,   tokenizer, max_length=max_length)
#test_dataset  = dataset.SequenceClassificationDataset(X_test,  y_test,  tokenizer, max_length=max_length)

In [97]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

## Model

In [98]:
model_cfg["vocab_size"] = tokenizer.vocab_size
model_cfg["num_classes"] = NUM_CLASSES
model_cfg["max_seq_len"] = max_length

In [99]:
model = sequence_classifier.SequenceClassifier(
    **model_cfg
)

In [100]:
optimizer = optimizer_factory.get_optimizer(optimizer_cfg, model)
criterion = loss_factory.get_loss_fn(loss_cfg)

## Sanity Check

In [101]:
sanity_checks.run_sanity_checks(model, train_loader, criterion, NUM_CLASSES)

Running sanity checks...
[OK] Output shape: (64, 10)
[--] Expected init loss: 2.3026 | Actual: 2.3381
[OK] Initial loss is reasonable.
[OK] All gradients flowing.
[--] Overfit final loss: 0.0004 (threshold: 0.1)
[--] Model weights restored.
[OK] Overfit single batch.
All sanity checks passed.


## Training Loop

### Cuda Check

In [102]:
if torch.cuda.is_available():
    print(f"CUDA is available. Training on GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("CUDA is not available. Training on CPU.")
    device = torch.device("cpu")

CUDA is available. Training on GPU: NVIDIA GeForce RTX 3070


### Logging Setup

In [103]:
log_dir = Path("../logs") / experiment_name
tb_dir  = log_dir / "tensorboard"
ckpt_dir  = log_dir / "checkpoints"

log_dir.mkdir(parents=True, exist_ok=True)
tb_dir.mkdir(parents=True, exist_ok=True)
ckpt_dir.mkdir(parents=True, exist_ok=True)

In [104]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "train.log"),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)

In [105]:
writer = SummaryWriter(tb_dir)

### Training Loop

In [106]:
model = model.to(device)

EPOCHS = 50

# -- For saving best checkpoint --
best_val_acc = 0.0

# -- For early stop --
best_val_loss = float("inf")
patience = 5
epochs_without_improvement = 0

for epoch in range(EPOCHS):

    # =========================================================
    # TRAINING
    # =========================================================
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)   # (B, num_classes)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_total   += labels.size(0)

    train_acc = train_correct / train_total

    # =========================================================
    # VALIDATION
    # =========================================================

    val_loss, val_acc = training.evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    # =========================================================
    # LOGGING
    # =========================================================

    logger.info(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f}"
    )

    writer.add_scalars(
        "loss",
        {
            "train": train_loss,
            "val": val_loss
        },
        epoch
    )

    writer.add_scalars(
        "accuracy",
        {
            "train": train_acc,
            "val": val_acc
        },
        epoch
    )

    # =========================================================
    # SAVING CHECKPOINT
    # =========================================================

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": model_cfg,
        "val_acc": val_acc,
    }

    torch.save(
        checkpoint,
        ckpt_dir / "last.pt"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save(
            checkpoint,
            ckpt_dir / "best.pt"
        )

        logger.info(
            f"Saved new best checkpoint "
            f"(val_acc={val_acc:.4f})"
        )

    # =========================================================
    # EARLY STOP
    # =========================================================
    improved = val_loss < best_val_loss

    if improved:
        best_val_loss = val_loss

        epochs_without_improvement = 0

        logger.info(
            f"Model improved "
            f"(val_loss={val_loss:.4f})"
        )

    else:
        epochs_without_improvement += 1

        logger.info(
            f"No improvement for "
            f"{epochs_without_improvement} epoch(s)"
        )

        if epochs_without_improvement > patience:
            logger.info(
                f"Activating stop loss on epoch {epoch}"
            )

            break


2026-05-29 20:05:43,280 | Epoch 1/50 | train_loss=355.5821 | train_acc=0.8029 | val_loss=0.2398 | val_acc=0.9286
2026-05-29 20:05:43,339 | Saved new best checkpoint (val_acc=0.9286)
2026-05-29 20:05:43,340 | Model improved (val_loss=0.2398)
2026-05-29 20:05:46,940 | Epoch 2/50 | train_loss=123.9113 | train_acc=0.9320 | val_loss=0.1562 | val_acc=0.9524
2026-05-29 20:05:46,990 | Saved new best checkpoint (val_acc=0.9524)
2026-05-29 20:05:46,991 | Model improved (val_loss=0.1562)
2026-05-29 20:05:50,624 | Epoch 3/50 | train_loss=86.4397 | train_acc=0.9527 | val_loss=0.1293 | val_acc=0.9606
2026-05-29 20:05:50,673 | Saved new best checkpoint (val_acc=0.9606)
2026-05-29 20:05:50,674 | Model improved (val_loss=0.1293)
2026-05-29 20:05:54,221 | Epoch 4/50 | train_loss=65.7319 | train_acc=0.9630 | val_loss=0.1180 | val_acc=0.9640
2026-05-29 20:05:54,269 | Saved new best checkpoint (val_acc=0.9640)
2026-05-29 20:05:54,270 | Model improved (val_loss=0.1180)
2026-05-29 20:05:57,810 | Epoch 5/50 |

In [107]:
# # loading
# checkpoint = torch.load("best.pt")
#
# model = SequenceClassifier(**checkpoint["config"])
#
# model.load_state_dict(
#     checkpoint["model_state_dict"]
# )
#
# optimizer.load_state_dict(
#     checkpoint["optimizer_state_dict"]
# )